In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Duomenų nuskaitymas
labels_data = pd.read_csv('LD2_dataset/labels.csv', dtype={'Image': str, 'class_label': int})
# Remove rows with class_label 5 or 8
labels_data = labels_data[~labels_data['class_label'].isin([5, 8])]

# Remap class labels: (0,2,4)->0, (3,6)->1, (1,7,9)->2
label_map = {0: 0, 2: 0, 4: 0, 3: 1, 6: 1, 1: 2, 7: 2, 9: 2}
labels_data['class_label'] = labels_data['class_label'].map(label_map)

labels_data['class_label'].value_counts().sort_index()

class_label
0    21000
1    14000
2    21000
Name: count, dtype: int64

In [3]:
train_val_df, test_df = train_test_split(labels_data, test_size=0.1, stratify=labels_data['class_label'], random_state=42)

# then split val from remaining (10% of total = ~11% of 90%)
train_df, val_df = train_test_split(train_val_df, test_size=0.111, stratify=train_val_df['class_label'], random_state=42)

print(len(train_df), len(val_df), len(test_df))  # ~48k, ~6k, ~6k
print('treniravimo aibe', train_df['class_label'].value_counts())
print('validacijos aibe', val_df['class_label'].value_counts())
print('testavimo aibe', test_df['class_label'].value_counts())

44805 5595 5600
treniravimo aibe class_label
2    16802
0    16802
1    11201
Name: count, dtype: int64
validacijos aibe class_label
0    2098
2    2098
1    1399
Name: count, dtype: int64
testavimo aibe class_label
2    2100
0    2100
1    1400
Name: count, dtype: int64


Pirma architektura NR 4

In [ ]:
class KetvirtaArchitektura(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3),  # (1, 28, 28) → (32, 26, 26)
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (32, 26, 26) → (32, 13, 13)
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3),  # (32, 13, 13) → (64, 11, 11)
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (64, 11, 11) → (64, 5, 5)
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3), # (64, 5, 5) → (128, 3, 3)
            nn.ReLU(),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),          # (128, 3, 3) → 1152
            nn.Linear(1152, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

5 architektura

In [ ]:
class PenktaArchitektura(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3),  # (1, 28, 28) → (32, 26, 26)
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (32, 26, 26) → (32, 13, 13)
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3),  # (32, 13, 13) → (64, 11, 11)
            nn.ReLU(),
            nn.AvgPool2d(2),                   # (64, 11, 11) → (64, 5, 5)
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3), # (64, 5, 5) → (128, 3, 3)
            nn.ReLU(),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),          # (128, 3, 3) → 1152
            nn.Dropout(p=0.4),        # Add dropout for regularization
            nn.Linear(1152, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

8 architektura

In [ ]:
class AstuntaArchitektura(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3),  # (1, 28, 28) → (32, 26, 26)
            nn.ReLU(),
            nn.AvgPool2d(2),                   # (32, 26, 26) → (32, 13, 13)
            nn.BatchNorm2d(32),                # Add batch normalization

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3),  # (32, 13, 13) → (64, 11, 11)
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (64, 11, 11) → (64, 5, 5)
            nn.BatchNorm2d(64),                # Add batch normalization
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),          # (128, 5, 5) → 1600
            nn.Linear(1600, 128),
            nn.BatchNorm1d(128),               # Add batch normalization
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
from tqdm import tqdm
from sklearn.metrics import f1_score

class DataFrameDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        # Assuming the 'Image' column contains filenames or prefixes
        img_val = str(self.df.iloc[idx]['Image'])
        if not img_val.endswith('.png') and not img_val.endswith('.jpg'):
            img_val += '.png'
            
        img_path = os.path.join(self.img_dir, img_val)
            
        image = Image.open(img_path).convert('L')  # Convert to grayscale
        label = int(self.df.iloc[idx]['class_label'])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

def train_model(model, model_name, train_df, val_df, img_dir='LD2_dataset/images', epochs=10, batch_size=32, lr=0.001, device=None):
    if device is None:
        device = torch.device('mps')
    print(f"Using device: {device}")
    
    best_weights_path = f"model_params/{model_name}_best_weights.pth"
    history_path = f"model_params/{model_name}_training_history.pkl"
 
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Generic transforms
    transform_train = transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    transform_val = transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    train_dataset = DataFrameDataset(train_df, img_dir, transform=transform_train)
    val_dataset = DataFrameDataset(val_df, img_dir, transform=transform_val)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    best_val_f1 = -1.0
    
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        all_train_preds = []
        all_train_labels = []
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())
            
            loop.set_postfix(loss=loss.item())
            
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        epoch_train_f1 = f1_score(all_train_labels, all_train_preds, average='macro')
        
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for inputs, labels in loop:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())
                
                loop.set_postfix(loss=loss.item())
                
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        epoch_val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')

        if epoch_val_f1 > best_val_f1:
            best_val_f1 = epoch_val_f1
            torch.save(model.state_dict(), best_weights_path)
            saved_msg = f" -> Saved new best weights to {best_weights_path} (Val F1: {epoch_val_f1:.4f})"
            best_epoch = epoch + 1
        else:
            saved_msg = ""
        
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['train_f1'].append(epoch_train_f1)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        history['val_f1'].append(epoch_val_f1)
        history['best_epoch'] = best_epoch
        print(f"Train Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.4f} | F1 Macro: {epoch_train_f1:.4f}")
        print(f"Val Loss: {epoch_val_loss:.4f} | Acc: {epoch_val_acc:.4f} | F1 Macro: {epoch_val_f1:.4f}{saved_msg}\n")
        

    with open(f'{history_path}', 'wb') as f:
        pickle.dump(history, f)
    return model, history

In [ ]:
import os
import matplotlib.pyplot as plt

def visualize_history(history, save_dir='/visualizations/train', model_name='model'):
    os.makedirs(save_dir, exist_ok=True)

    epochs = range(1, len(history['train_loss']) + 1)
    best_epoch = history.get('best_epoch', None)

    def mark_best(ax, best_epoch, y_values):
        if best_epoch is not None:
            ax.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.6, label=f'Best epoch ({best_epoch})')
            ax.plot(best_epoch, y_values[best_epoch - 1], 'r*', markersize=12, zorder=5)

    # 1. Loss
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history['train_loss'], label='Train Loss')
    ax.plot(epochs, history['val_loss'], label='Val Loss')
    mark_best(ax, best_epoch, history['val_loss'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Train vs Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{model_name}_loss.png'), dpi=150)
    plt.close(fig)

    # 2. Accuracy
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history['train_acc'], label='Train Accuracy')
    ax.plot(epochs, history['val_acc'], label='Val Accuracy')
    mark_best(ax, best_epoch, history['val_acc'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Train vs Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{model_name}_accuracy.png'), dpi=150)
    plt.close(fig)

    # 3. F1
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history['train_f1'], label='Train F1')
    ax.plot(epochs, history['val_f1'], label='Val F1')
    mark_best(ax, best_epoch, history['val_f1'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('F1 (macro)')
    ax.set_title('Train vs Validation F1')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{model_name}_f1.png'), dpi=150)
    plt.close(fig)

    # 4. Val F1 and Val Accuracy together
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history['val_f1'], label='Val F1')
    ax.plot(epochs, history['val_acc'], label='Val Accuracy')
    mark_best(ax, best_epoch, history['val_f1'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Validation F1 vs Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, f'{model_name}_val_f1_vs_acc.png'), dpi=150)
    plt.close(fig)


In [ ]:
import os
import json
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def test_model(model, model_name, metrics_save_dir, plots_save_dir, test_df, img_dir='LD2_dataset/images', device=None, batch_size=32):
    if device is None:
        device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
        
    os.makedirs(metrics_save_dir, exist_ok=True)
    os.makedirs(plots_save_dir, exist_ok=True)
    
    transform_test = transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    # Reuse DataFrameDataset for loading test data
    test_dataset = DataFrameDataset(test_df, img_dir, transform=transform_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Evaluate and compare to test_df values
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    
    metrics = {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'precision_macro': precision,
        'recall_macro': recall
    }
    
    # Output metrics to dictionary
    metrics_path = os.path.join(metrics_save_dir, f"{model_name}_metrics.json")
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)
        
    # Create the confusion matrix and save as PNG
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    
    cm_plot_path = os.path.join(plots_save_dir, f"{model_name}_confusion_matrix.png")
    plt.savefig(cm_plot_path, dpi=150)
    plt.close()
    
    return metrics


In [ ]:
import torch

def load_trained_model(model_class, weights_path, num_classes=3, device=None):
    if device is None:
        device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
    
    # Instantiate the model architecture
    model = model_class(num_classes=num_classes)
    
    # Load the state dictionary
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    
    model.to(device)
    model.eval()
    
    print(f"Model {model_class.__name__} loaded successfully from {weights_path}")
    return model
